# ComfyUI + LTX-2.5 Image-to-Video — KaggleWorkflow: `video_ltx2_5_i2v-kaggle.json` (Image-to-Video de Lightricks).Usa Cloudflare Tunnel (no necesita token ni cuenta).Modelo de video LTX-2.5 22B INT8 + audio sincronizado.

In [ ]:
import os, shutil, subprocess, time, requests

BASE = os.getcwd()  # /kaggle/working
# TODO el ComfyUI + modelos en /kaggle/temp (más espacio)
TEMP_COMFY = "/kaggle/temp/ComfyUI"
print("Directorio de trabajo:", BASE)
print("ComfyUI + modelos en:", TEMP_COMFY)

# Verificar espacio
import shutil as sh
for p in ["/kaggle/working", "/kaggle/temp"]:
    try:
        total, used, free = sh.disk_usage(p)
        print(f"  {p}: total={total//2**30}GB usado={used//2**30}GB libre={free//2**30}GB")
    except Exception as e:
        print(f"  {p}: no disponible ({e})")

# Limpiar
if os.path.exists(TEMP_COMFY):
    shutil.rmtree(TEMP_COMFY)

# Clonar ComfyUI en /kaggle/temp
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /kaggle/temp/ComfyUI
os.chdir(TEMP_COMFY)

# Crear directorios de modelos
for sub in ["diffusion_models", "text_encoders", "vae", "latent_upscale_models", "loras"]:
    os.makedirs(os.path.join(TEMP_COMFY, "models", sub), exist_ok=True)
os.makedirs(os.path.join(TEMP_COMFY, "input"), exist_ok=True)

!pip install -q -r requirements.txt

print("Instalando huggingface_hub...")
!pip install -q huggingface_hub

from huggingface_hub import hf_hub_download

REPO = "comfyicu/LTX-2.5"

def download_hf(repo, path, dest_dir):
    os.makedirs(dest_dir, exist_ok=True)
    name = os.path.basename(path)
    dest = os.path.join(dest_dir, name)
    if os.path.exists(dest) and os.path.getsize(dest) > 10**6:
        print(f"  ✓ {name} ya existe ({os.path.getsize(dest)//2**30} GB), omitir")
        return
    print(f"  Descargando {name}...")
    try:
        hf_hub_download(repo_id=repo, filename=path, local_dir=dest_dir, local_dir_use_symlinks=False)
        size = os.path.getsize(dest) if os.path.exists(dest) else 0
        print(f"    ✓ {name}: {size//2**30} GB")
    except Exception as e:
        print(f"    ✗ Error descargando {name}: {e}")
        raise

# 1. Diffusion model (21.5 GB)
download_hf(REPO, "diffusion_models/ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors", os.path.join(TEMP_COMFY, "models/diffusion_models"))

# 2. Video VAE (1.5 GB)
download_hf(REPO, "vae/ltx-2.5-video-vae-bf16.safetensors", os.path.join(TEMP_COMFY, "models/vae"))

# 3. Audio VAE (365 MB)
download_hf(REPO, "vae/ltx-2.5-audio-vae-bf16.safetensors", os.path.join(TEMP_COMFY, "models/vae"))

# 4. Text encoder Gemma 12B (15.4 GB)
download_hf(REPO, "text_encoders/gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors", os.path.join(TEMP_COMFY, "models/text_encoders"))

# 5. Latent upscaler (1 GB)
download_hf(REPO, "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors", os.path.join(TEMP_COMFY, "models/latent_upscale_models"))

# 6. Imagen de entrada
!curl -L -o input/neon_cyborg_portrait.png https://raw.githubusercontent.com/Comfy-Org/workflow_templates/refs/heads/main/input/neon_cyborg_portrait.png

print("\n=== Modelos descargados ===\n")
total = 0
for root, dirs, files in os.walk(os.path.join(TEMP_COMFY, "models")):
    for f in files:
        fp = os.path.join(root, f)
        size = os.path.getsize(fp)
        total += size
        print(f"  {f}: {size//2**30} GB ({size//2**20} MB)")
print(f"\nTotal: {total//2**30} GB")

print("\nModelos listos. Iniciando ComfyUI...")

log = open("comfyui.log", "w")
proc = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],    cwd=TEMP_COMFY, stdout=log, stderr=log
)
print("ComfyUI PID:", proc.pid)

for i in range(120):
    try:
        r = requests.get("http://localhost:8188", timeout=5)
        if r.status_code == 200:
            print("ComfyUI listo en :8188")
            break
    except:
        pass
    time.sleep(1)
else:
    print("ComfyUI no arranco. Revisa comfyui.log:")
    print(open("comfyui.log").read()[-1500:])

In [ ]:
print("Instalando Cloudflare Tunnel...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

cf_log = open("cloudflare.log", "w")
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=cf_log, stderr=cf_log
)
time.sleep(8)

log_content = open("cloudflare.log").read()
import re
match = re.search(r'https://[a-z-]+\.trycloudflare\.com', log_content)
if match:
    url = match.group()
    print("=" * 60)
    print(f"ComfyUI en linea!")
    print(f"Enlace: {url}")
    print("=" * 60)
else:
    print("No se encontro la URL. Revisa cloudflare.log:")
    print(log_content[-1500:])

## Workflow: Image-to-Video (LTX-2.5)
- Carga el flujo: `video_ltx2_5_i2v-kaggle.json`
- Arrastra el JSON a ComfyUI para cargarlo

## Modelos descargados
| Tipo | Archivo | Nota |
|------|---------|------|
| Diffusion | `ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors` | ~22B INT8 |
| Video VAE | `ltx-2.5-video-vae-bf16.safetensors` | Decodificador de video |
| Audio VAE | `ltx-2.5-audio-vae-bf16.safetensors` | Audio sincronizado |
| Text encoder | `gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors` | Gemma 4 12B |
| Latent upscaler | `ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors` | Upscaler x2 |
| Input | `neon_cyborg_portrait.png` | Primer frame de ejemplo |

## Uso
1. Carga una imagen en **Load First Frame** (o usa la de ejemplo)
2. Escribe un prompt describiendo el movimiento
3. Ajusta duration, width, height, frame_rate
4. Mantener `prompt_enhance` desactivado (no se descargó el modelo enhancer)5. Pulsa **Queue**

## Notas
- Cierra el tunnel en Kaggle: `File` -> `Stop execution`
- GPU: T4 x2 (32 GB VRAM) - ComfyUI usa una GPU automáticamente- El modelo de difusión (~22 GB INT8) puede tardar en descargarse